# Cox survival analysis

* `mydir`: path to the input data.
* `ostime_column`: survival duration column; other endpoints may be used instead of overall survival.
* `os`: event indicator; other survival endpoints may be used instead of overall survival.

In [ ]:
from lifelines import CoxPHFitter
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df
from sklearn.model_selection import train_test_split
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.comp1 import fillna

survival_type = get_param_in_cwd('event_col', 'event')
task_type = f'Clinical_'
event_col = survival_type
duration_col = get_param_in_cwd('duration_col', 'duration')
data = pd.read_csv('data/clinical.csv', dtype={'ID':str})
survival_data = pd.read_csv(get_param_in_cwd('label_file'))
data = pd.merge(data[[c for c in data.columns if c not in ['label']]], survival_data[['ID', event_col, duration_col]], on='ID', how='inner')
# data = normalize_df(data, not_norm=['ID', 'group'])
# label_data = pd.read_csv(get_param_in_cwd('label_file'), dtype={'ID':str})[['ID', 'group']]
# data = pd.merge(data, label_data, on='ID', how='inner')
data

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp
from collections import OrderedDict

group_info = 'group'
train_data = data[(data[group_info] == 'train')]

# subsets = [s for s in label_data['group'].value_counts().index if s != 'train']
subsets = get_param_in_cwd('subsets')
val_datasets = OrderedDict()
for subset in subsets:
    val_data = data[data[group_info] == subset]
    val_datasets[subset] = val_data
    val_data.to_csv(f'features/{task_type}{subset}_features_norm.csv', index=False)

print(', '.join([f"{subset} sample dimensions: {d_.shape}" for subset, d_ in val_datasets.items()]))

In [ ]:
import os
from onekey_algo.custom.components.survival import uni_cox

if os.path.exists(f'features/{task_type}rad_features_unisel.csv') and False:
    train_data = pd.read_csv(f'features/{task_type}rad_features_unisel.csv')
else:
    sel_features, uni_info = uni_cox(train_data, duration_col=duration_col, event_col=event_col,
                                     cols=[c for c in train_data.columns if c not in [event_col, duration_col, 'ID', 'group']], 
                                     pvalue_thres=0.05, verbose=True)
    train_data = train_data[['ID'] + sel_features + [event_col, duration_col, 'group']]
    data[['ID'] + sel_features + [event_col, duration_col, 'group']].to_csv(f'features/{task_type}rad_features_unisel.csv', header=True, index=False)
train_data

## Cox regression summary

Key quantities to inspect:
1. `Concordance`: concordance index (C-index).
2. `exp(coef)`: feature hazard ratio (HR), reported with a 95% confidence interval.
3. `p`: p-value for the feature coefficient.

In [ ]:
train_data[[c for c in train_data.columns if c != 'ID']]

In [ ]:
from lifelines import CoxPHFitter
mci = 0
max_p = 0
for p in range(1, 100):
    cph = CoxPHFitter(penalizer=p/100)
    cph.fit(train_data[[c for c in train_data.columns if c not in ['ID', 'group']]], duration_col=duration_col, event_col=event_col)
    test_data = val_datasets['test']
    ci = cph.score(test_data[[c for c in test_data.columns if c not in ['ID', 'group']]], scoring_method="concordance_index")
    if mci < ci:
        print(p, ci)
        max_p = p
        mci = ci
cph = CoxPHFitter(penalizer=47/100)
cph.fit(train_data[[c for c in train_data.columns if c not in ['ID', 'group']]], duration_col=duration_col, event_col=event_col)
cph.print_summary()

#### Report the hazard ratio for each feature

In [ ]:
multi_info = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].reset_index()
i = pd.merge(uni_info, multi_info, on='covariate', how='left', suffixes=['_Uni', '_Multi'])
multi_data = data[['ID'] + list(multi_info[multi_info['p'] < 0.05]['covariate'])]
multi_data.to_csv(f'features/{task_type}rad_features_mulsel.csv', header=True, index=False)
i['p_Uni'] = i['p_Uni'].map(lambda x: '<0.05' if not pd.isna(x) and x < 0.05 else x)
i['p_Multi'] = i['p_Multi'].map(lambda x: '<0.05' if not pd.isna(x) and x < 0.05 else x)
i.columns = ['features_name', 'HR', 'lower 95%CI', 'upper 95%CI', 'pvalue', 'HR', 'lower 95%CI', 'upper 95%CI', 'pvalue']
i.applymap(lambda x: '' if pd.isna(x) else x)

In [ ]:
data[['ID'] + list(cph.summary[cph.summary['p'] < 0.05].index) + [event_col, duration_col, 'group']].to_csv(f'data/{task_type}clinical_sur.csv', index=False)
data[['ID'] + list(cph.summary[cph.summary['p'] < 0.05].index) + [event_col, duration_col, 'group']]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
cph.plot(hazard_ratios=True)
plt.savefig('img/Clinic_feature_pvalue.svg')
plt.show()

# Kaplan-Meier curves

Stratify patients by predicted hazard and calculate Kaplan-Meier estimates and log-rank tests.

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

thres = 0.001
bst_split = {'test2': 0.9339496559686359,}
loc = { 'test1': 0.7, 'val': 0.7, 'train':0.6, 'test2':0.75}
for subset, test_data in val_datasets.items():
    c_index = cph.score(test_data[[c for c in test_data.columns if c != 'ID']], scoring_method="concordance_index")
#     y_pred = cph.predict_median(test_data[[c for c in test_data.columns if c != 'ID']])
#     cox_data = pd.concat([test_data, y_pred], axis=1)
#     mean = cox_data.describe()[0.5]['mean']
#     cox_data['HR'] = cox_data[0.5] < mean
    y_pred = cph.predict_partial_hazard(test_data[[c for c in test_data.columns if c != 'ID']])
    cox_data = pd.concat([test_data, y_pred], axis=1)

    p = 1
    cutoff = None
    for i in sorted(list(cox_data[0])):
        cox_data['HR'] = cox_data[0] > i
        dem = (cox_data["HR"] == True)
        results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                               event_observed_A=cox_data[event_col][dem], event_observed_B=cox_data[event_col][~dem])
        if p > results.p_value:
            # print(i, results.p_value)
            p = results.p_value
            cutoff = i
            # if results.p_value < thres:
            #     break
#         if subset == 'test2':
#             print(cutoff, results.p_value)
    # mean = cox_data.describe()[0]['50%']
    # cox_data['HR'] = cox_data[0] > bst_split[subset] if subset in bst_split else 1
#     print(subset, cutoff, mean)
    cox_data['HR'] = cox_data[0] > cutoff
    dem = (cox_data["HR"] == True)
    results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                           event_observed_A=cox_data[event_col][dem], event_observed_B=cox_data[event_col][~dem])
    p_value = f"={results.p_value:.3f}" if results.p_value > thres else f'<{thres}'
    plt.title(f"Cohort {subset} C-index:{c_index:.3f}")
    plt.ylabel('Probability')
    if sum(dem):
        kmf_high = KaplanMeierFitter()
        kmf_high.fit(cox_data[duration_col][dem], event_observed=cox_data[event_col][dem], label="High Risk")
        kmf_high.plot_survival_function(color='r')
    if sum(~dem):
        kmf_low = KaplanMeierFitter()
        kmf_low.fit(cox_data[duration_col][~dem], event_observed=cox_data[event_col][~dem], label="Low Risk")
        kmf_low.plot_survival_function(color='g')
    plt.legend(loc=3)
    y_min, y_max = plt.ylim()
    plt.text(0.5, y_min + (y_max-y_min) / 3, f"P{p_value}")
    plt.xlabel('Time(months)')
    add_at_risk_counts(kmf_high, kmf_low, rows_to_show=['At risk'])
    plt.savefig(f'img/{task_type}KM_{subset}.svg', bbox_inches='tight')
    plt.show()

In [ ]:
import os
import numpy as np

def get_prediction(model: CoxPHFitter, data, ID=None, **kwargs):
    hr = model.predict_partial_hazard(data)
    expectation = model.predict_expectation(data)
    
    predictions = pd.concat([hr, expectation], axis=1)
    predictions.columns = ['HR', 'expectation']
    if ID is not None:
        predictions = pd.concat([ID, hr, expectation], axis=1)
        predictions.columns = ['ID', 'HR', 'expectation']
    else:
        predictions = pd.concat([hr, expectation], axis=1)
        predictions.columns = ['HR', 'expectation']
    return predictions

os.makedirs('results', exist_ok=True)
info = []
for subset, test_data in val_datasets.items():
    if subset in get_param_in_cwd('subsets') + ['TCGA']:
        results = get_prediction(cph, test_data, ID=test_data['ID'])
        results.to_csv(f'results/{task_type}cox_predictions_{subset}.csv', index=False)
        results['group'] = subset
        info.append(results)
        pd.merge(results, data[['ID', event_col, duration_col]], 
                 on='ID', how='inner').to_csv(f'features/{task_type}4xtile_{subset}.txt', index=False, sep='\t')
info = pd.concat(info, axis=0)
info

In [ ]:
# info = pd.merge(data[['ID', event_col, duration_col]], info[['ID', 'group']], on='ID', how='inner')
# info.to_csv('group.csv', index=False)
# info